# Notebook 01 — Ingestion Pipeline Observability

This notebook observes the **Phase 1 knowledge ingestion pipeline** by hitting the live FastAPI endpoints.

What we will inspect:
1. Raw passages from the HuggingFace dataset
2. Trigger and observe the ingestion process
3. Inspect embedding dimensionality
4. Verify the vector store was persisted to disk

**Prerequisite:** the API server must be running.
```
uv run uvicorn main:app --app-dir src --port 8004
```

In [1]:
import requests
import pandas as pd
from IPython.display import display

BASE_URL = "http://localhost:8004"

resp = requests.get(f"{BASE_URL}/health")
print("Server status:", resp.json()["status"])
print("Version      :", resp.json()["version"])

Server status: ok
Version      : 0.1.0


## 1. Browse Raw Passages

Inspect sample passages from the HuggingFace dataset before ingesting them.

In [2]:
resp = requests.get(f"{BASE_URL}/passages?limit=5")
body = resp.json()

print(f"Total passages in dataset: {body['total']:,}")
print(f"Showing first {len(body['passages'])} passages:\n")
for p in body["passages"]:
    print(f"  [{p['id']}] {p['text'][:120]}...")
    print()

Total passages in dataset: 3,200
Showing first 5 passages:

  [0] Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the southeastern part o...

  [1] It is bordered by Brazil to the north, by Argentina across the bank of both the Uruguay River to the west and the estuar...

  [2] Montevideo was founded by the Spanish in the early 18th century as a military stronghold. Uruguay won its independence i...

  [3] The economy is largely based in agriculture (making up 10% of the GDP and the most substantial export) and the state-sec...

  [4] According to Transparency International, Uruguay is the second least corrupt country in Latin America (after Chile),  Tr...



In [3]:
# Display as a DataFrame for easier inspection
rows = [{"id": p["id"], "text_preview": p["text"][:100] + "..."} for p in body["passages"]]
display(pd.DataFrame(rows))

,id,text_preview
0,0,"Uruguay (official full name in ; pron. , Eas..."
1,1,"It is bordered by Brazil to the north, by Arge..."
2,2,Montevideo was founded by the Spanish in the e...
3,3,The economy is largely based in agriculture (m...
4,4,"According to Transparency International, Urugu..."


## 2. Trigger Ingestion

`POST /ingest?limit=N` — embeds passages with Azure `text-embedding-3-large` and builds the local vector index.
Use `limit` to keep it fast during development. Remove it for a full ingest.

In [6]:
resp = requests.post(f"{BASE_URL}/ingest?limit=50")
status = resp.json()

print("Ingestion result:")
for k, v in status.items():
    print(f"  {k:20s}: {v}")

Ingestion result:
  status              : ready
  passage_count       : 50
  index_path          : C:\Users\jcampbell\Downloads\forge\da-rag-project-jlcampbell12\data\vector_store
  index_on_disk       : True


## 3. Ingestion Status

Poll the status endpoint to see how many passages are indexed and where the vector store lives on disk.

In [7]:
resp = requests.get(f"{BASE_URL}/ingest/status")
status = resp.json()

print("Vector store status:")
for k, v in status.items():
    print(f"  {k:20s}: {v}")

Vector store status:
  status              : ready
  passage_count       : 50
  index_path          : C:\Users\jcampbell\Downloads\forge\da-rag-project-jlcampbell12\data\vector_store
  index_on_disk       : True


## 4. Inspect the Vector Store on Disk

List the files persisted by LlamaIndex to confirm the index was saved correctly.

In [8]:
import os

index_path = status["index_path"]
if os.path.exists(index_path):
    files = os.listdir(index_path)
    total_size = sum(
        os.path.getsize(os.path.join(index_path, f)) for f in files
    )
    print(f"Index directory: {index_path}")
    print(f"Total size on disk: {total_size / 1024:.1f} KB")
    print(f"\nFiles ({len(files)}):")
    for f in sorted(files):
        size = os.path.getsize(os.path.join(index_path, f))
        print(f"  {f:40s}  {size / 1024:6.1f} KB")
else:
    print("Index directory not found — run the ingest cell above first.")

Index directory: C:\Users\jcampbell\Downloads\forge\da-rag-project-jlcampbell12\data\vector_store
Total size on disk: 3457.7 KB

Files (5):
  default__vector_store.json                3380.6 KB
  docstore.json                               72.7 KB
  graph_store.json                             0.0 KB
  image__vector_store.json                     0.1 KB
  index_store.json                             4.3 KB


## 5. Embedding Dimensionality

Verify the embedding model produces 3072-dimensional vectors (as expected for `text-embedding-3-large`).

In [9]:
resp = requests.post(f"{BASE_URL}/embed?query=test+embedding+dimensions")
body = resp.json()

print(f"Query      : {body['query']}")
print(f"Dimensions : {body['dimensions']}")
print(f"First 5 values: {[round(v, 6) for v in body['embedding'][:5]]}")
print(f"\nNote: text-embedding-3-large produces {body['dimensions']}-dim vectors")

Query      : test embedding dimensions
Dimensions : 3072
First 5 values: [-0.013752, 0.000984, -0.015333, 0.005751, 0.035328]

Note: text-embedding-3-large produces 3072-dim vectors
